# 08 - SAINT | Sensor 1

Feature-tokenizer transformer with **intersample attention** (true SAINT).

Trained from scratch on this sensor. 70/15/15 split; validation picks the best epoch, test used once.

### Cell 1 - Imports

In [1]:
import os
import copy
import random
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

### Cell 2 - Device

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", DEVICE)
if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))

Device : cuda
GPU    : NVIDIA GeForce RTX 3050 6GB Laptop GPU


### Cell 3 - Random seed

In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

### Cell 4 - Load Sensor 1

In [4]:
df = pd.read_csv("../data/processed/sensor1_processed.csv")
print(df.shape)
df.head()

(4895158, 15)


,Dur,Proto,Sport,Dir,Dport,State,sTos,dTos,TotPkts,TotBytes,SrcBytes,Hour,Minute,Second,ActivityLabel
0,-0.382178,2.0,119.0,0.0,13395.0,344.0,-0.016359,-0.012325,-0.017028,-0.014875,-0.014910,-1.373052,-1.686039,-1.680628,1
1,-0.382177,13.0,51465.0,3.0,64513.0,17.0,-0.016359,-0.012325,-0.016801,-0.014869,-0.014956,-1.373052,-1.686039,-1.680628,0
2,-0.382177,13.0,57568.0,3.0,64513.0,17.0,-0.016359,-0.012325,-0.016801,-0.014869,-0.014956,-1.373052,-1.686039,-1.680628,0
3,3.028843,13.0,14263.0,3.0,20844.0,17.0,-0.016359,-0.012325,-0.015442,-0.014686,-0.014690,-1.373052,-1.686039,-1.680628,0
4,-0.382177,13.0,40041.0,3.0,64513.0,17.0,-0.016359,-0.012325,-0.016801,-0.014869,-0.014956,-1.373052,-1.686039,-1.680628,0


### Cell 5 - Features and labels

In [5]:
TARGET = "ActivityLabel"

X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].copy()

print("Features :", X.shape[1])
print("Samples  :", X.shape[0])
print("Classes  :", y.nunique())
display(y.value_counts())

Features : 14
Samples  : 4895158
Classes  : 2


### Cell 6 - Encode labels

In [6]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

NUM_CLASSES = len(label_encoder.classes_)
print("Number of classes:", NUM_CLASSES)

Number of classes: 2


### Cell 7 - Train / validation / test split

In [7]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print("Train :", X_train.shape)
print("Valid :", X_val.shape)
print("Test  :", X_test.shape)

Train : (3426610, 14)
Valid : (734274, 14)
Test  : (734274, 14)


### Cell 8 - Scale features (fit on train only)
The continuous tokenizer is sensitive to feature magnitude, so standardize.

In [8]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype("float32")
X_val   = scaler.transform(X_val).astype("float32")
X_test  = scaler.transform(X_test).astype("float32")

print(X_train.dtype, X_train.shape)

float32 (3426610, 14)


### Cell 9 - PyTorch Dataset

In [9]:
class SensorDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

### Cell 10 - DataLoaders

In [10]:
BATCH_SIZE = 512

train_dataset = SensorDataset(X_train, y_train)
val_dataset   = SensorDataset(X_val,   y_val)
test_dataset  = SensorDataset(X_test,  y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print("Train batches:", len(train_loader))

Train batches: 6693


### Cell 11 - Model config
Dimensions derived from the data - no hardcoding.

In [11]:
NUM_FEATURES = X_train.shape[1]
NUM_TOKENS   = NUM_FEATURES + 1   # +1 for the CLS token

EMBED_DIM  = 64
NUM_HEADS  = 8
NUM_LAYERS = 3
FFN_DIM    = 256
DROPOUT    = 0.2

assert EMBED_DIM % NUM_HEADS == 0
print("Features:", NUM_FEATURES, "| Classes:", NUM_CLASSES)

Features: 14 | Classes: 2


### Cell 12 - Numerical feature tokenizer
`(B, F)` -> `(B, F, D)`: each scalar feature becomes a D-dim token.

In [12]:
class NumericalFeatureTokenizer(nn.Module):
    def __init__(self, num_features, embed_dim):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(num_features, embed_dim))
        self.bias   = nn.Parameter(torch.randn(num_features, embed_dim))

    def forward(self, x):
        x = x.unsqueeze(-1)              # (B, F, 1)
        return x * self.weight + self.bias   # (B, F, D)

### Cell 13 - Feed forward network

In [13]:
class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

### Cell 14 - SAINT block (column self-attention + intersample attention)
This is what makes it SAINT rather than a plain FT-Transformer: after the
usual feature (column) self-attention, an **intersample** attention lets each
row attend to the other rows in the batch.

In [14]:
class SAINTBlock(nn.Module):
    def __init__(self, dim, num_tokens, heads, ff_dim, dropout):
        super().__init__()
        # Column (feature) self-attention
        self.col_norm    = nn.LayerNorm(dim)
        self.col_attn    = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.col_ff_norm = nn.LayerNorm(dim)
        self.col_ff      = FeedForward(dim, ff_dim, dropout)

        # Row (intersample) attention over flattened per-sample tokens
        row_dim = num_tokens * dim
        self.row_norm    = nn.LayerNorm(dim)
        self.row_attn    = nn.MultiheadAttention(row_dim, heads, dropout=dropout, batch_first=True)
        self.row_ff_norm = nn.LayerNorm(dim)
        self.row_ff      = FeedForward(dim, ff_dim, dropout)

    def forward(self, x):
        # x: (B, N, D)
        h = self.col_norm(x)
        a, _ = self.col_attn(h, h, h, need_weights=False)
        x = x + a
        x = x + self.col_ff(self.col_ff_norm(x))

        b, n, d = x.shape
        h = self.row_norm(x).reshape(1, b, n * d)   # one sequence of b samples
        a, _ = self.row_attn(h, h, h, need_weights=False)
        x = x + a.reshape(b, n, d)
        x = x + self.row_ff(self.row_ff_norm(x))
        return x

### Cell 15 - SAINT model

In [15]:
class SAINT(nn.Module):
    def __init__(self, num_features, num_classes, embed_dim,
                 num_heads, num_layers, ff_dim, dropout=0.2):
        super().__init__()
        num_tokens = num_features + 1

        self.tokenizer = NumericalFeatureTokenizer(num_features, embed_dim)
        self.feature_embedding = nn.Parameter(torch.randn(1, num_features, embed_dim))
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.dropout = nn.Dropout(dropout)

        self.encoder = nn.ModuleList([
            SAINTBlock(embed_dim, num_tokens, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, num_classes),
        )
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.normal_(self.cls_token, std=0.02)
        nn.init.normal_(self.feature_embedding, std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.tokenizer(x)                 # (B, F, D)
        x = x + self.feature_embedding

        B = x.size(0)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)        # (B, F+1, D)
        x = self.dropout(x)

        for block in self.encoder:
            x = block(x)

        x = self.final_norm(x)
        cls_representation = x[:, 0]
        return self.head(cls_representation)

### Cell 16 - Build model

In [16]:
model = SAINT(
    num_features=NUM_FEATURES,
    num_classes=NUM_CLASSES,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    ff_dim=FFN_DIM,
    dropout=DROPOUT,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print("Parameters:", f"{n_params:,}")

Parameters: 11,340,738


### Cell 17 - Loss, optimizer, scheduler
Class weights help if NCC2 is imbalanced - remove `weight=` if not needed.

In [17]:
EPOCHS = 30

counts  = np.bincount(y_train, minlength=NUM_CLASSES)
weights = torch.tensor(counts.sum() / (NUM_CLASSES * counts), dtype=torch.float32).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

### Cell 18 - Epoch runner

In [18]:
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, n = 0.0, 0, 0
    torch.set_grad_enabled(train)
    for inputs, labels in loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        if train:
            optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        if train:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        n += labels.size(0)
    torch.set_grad_enabled(True)
    return total_loss / n, correct / n

### Cell 19 - Train loop (keeps best-val weights)

In [19]:
best_val_acc, best_state = 0.0, None
patience, wait = 6, 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    torch.set_grad_enabled(True)      # guards against a stale no_grad leaking in
    tr_loss, tr_acc, seen = 0.0, 0, 0
    for bi, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(inputs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item() * labels.size(0)
        tr_acc  += (out.argmax(1) == labels).sum().item()
        seen    += labels.size(0)

    val_loss, val_acc = run_epoch(val_loader, train=False)

    if val_acc > best_val_acc:
        best_val_acc, best_state, wait = val_acc, copy.deepcopy(model.state_dict()), 0
    else:
        wait += 1

    print(f"Epoch {epoch:03d} | train {tr_loss/seen:.4f}/{tr_acc/seen:.4f} | "
          f"val {val_loss:.4f}/{val_acc:.4f}")

    if wait >= patience:
        print("Early stop at epoch", epoch)
        break

print("Best val acc:", round(best_val_acc, 4))

Epoch 001 | train 0.2301/0.8884 | val 0.2026/0.8968
Epoch 002 | train 0.1873/0.9004 | val 0.1719/0.8893
Epoch 003 | train 0.1744/0.9065 | val 0.1577/0.9100
Epoch 004 | train 0.1683/0.9088 | val 0.1592/0.9041
Epoch 005 | train 0.1638/0.9114 | val 0.1494/0.9123
Epoch 006 | train 0.1604/0.9128 | val 0.1531/0.9275
Epoch 007 | train 0.1570/0.9140 | val 0.1433/0.9211
Epoch 008 | train 0.1544/0.9153 | val 0.1483/0.9194
Epoch 009 | train 0.1515/0.9169 | val 0.1440/0.9144
Epoch 010 | train 0.1500/0.9174 | val 0.1404/0.9251
Epoch 011 | train 0.1473/0.9185 | val 0.1567/0.9336
Epoch 012 | train 0.1452/0.9204 | val 0.1434/0.9243
Epoch 013 | train 0.1427/0.9213 | val 0.1455/0.9365
Epoch 014 | train 0.1416/0.9221 | val 0.1296/0.9289
Epoch 015 | train 0.1388/0.9237 | val 0.1284/0.9202
Epoch 016 | train 0.1370/0.9249 | val 0.1222/0.9256
Epoch 017 | train 0.1350/0.9256 | val 0.1309/0.9335
Epoch 018 | train 0.1329/0.9269 | val 0.1376/0.9352
Epoch 019 | train 0.1307/0.9279 | val 0.1383/0.9259
Early stop a

### Cell 20 - Predict on test set

In [20]:
model.load_state_dict(best_state)
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs.to(DEVICE))
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

### Cell 21 - Accuracy, precision, recall, F1

In [21]:
acc  = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
rec  = recall_score(all_labels, all_preds, average="weighted", zero_division=0)
f1_w = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
f1_m = f1_score(all_labels, all_preds, average="macro", zero_division=0)

print(f"Accuracy      : {acc:.4f}")
print(f"Precision (w) : {prec:.4f}")
print(f"Recall (w)    : {rec:.4f}")
print(f"F1 (weighted) : {f1_w:.4f}")
print(f"F1 (macro)    : {f1_m:.4f}")

Accuracy      : 0.9360
Precision (w) : 0.9778
Recall (w)    : 0.9360
F1 (weighted) : 0.9511
F1 (macro)    : 0.7174


### Cell 22 - Classification report

In [22]:
target_names = [str(c) for c in label_encoder.classes_]
print(classification_report(all_labels, all_preds,
                            target_names=target_names, digits=4))

              precision    recall  f1-score   support

           0     0.9983    0.9357    0.9660    712374
           1     0.3115    0.9467    0.4688     21900

    accuracy                         0.9360    734274
   macro avg     0.6549    0.9412    0.7174    734274
weighted avg     0.9778    0.9360    0.9511    734274



### Cell 23 - Confusion matrix

In [23]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=target_names, yticklabels=target_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("SAINT - Sensor 1 Confusion Matrix")
plt.tight_layout()
plt.show()